# <font color = 'red'> Dependencias

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel

import sys
import os

# Agregar la carpeta calibration_code al path
sys.path.append(os.path.abspath("../../calibration_code"))

# Ahora puedes importar los módulos personalizados
from modelling_tools import (plot_histogram, plot_univariate_freq, assign_deciles, count_categories_by_decile, 
                             calculate_category_proportions, summarize_decile_analysis, summarize_grouped_deciles, group_deciles,
                             compute_odds_ratio, count_by_category, proportions_by_category, counts_and_proportions_by_category)
from visualization_tools import plot_interactive_chart, plot_categorical_proportions
from utils import g
from config import get_data_path, get_code_path
from data_cleaning import check_dataframe_quality

# <font color = 'red'> Carga de Datos

In [2]:
df = pd.read_csv(get_data_path("preprocessed_data.csv"))

df = df.filter(regex = 'Occupation|Credit_Mix|Credit_Score').drop('Binary_Credit_Score', axis = 1)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 17 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   Credit_Mix                100000 non-null  object
 1   Credit_Score              100000 non-null  int64 
 2   Occupation_Accountant     100000 non-null  int64 
 3   Occupation_Architect      100000 non-null  int64 
 4   Occupation_Developer      100000 non-null  int64 
 5   Occupation_Doctor         100000 non-null  int64 
 6   Occupation_Engineer       100000 non-null  int64 
 7   Occupation_Entrepreneur   100000 non-null  int64 
 8   Occupation_Journalist     100000 non-null  int64 
 9   Occupation_Lawyer         100000 non-null  int64 
 10  Occupation_Manager        100000 non-null  int64 
 11  Occupation_Mechanic       100000 non-null  int64 
 12  Occupation_Media_Manager  100000 non-null  int64 
 13  Occupation_Musician       100000 non-null  int64 
 14  Occup

In [3]:
res = check_dataframe_quality(df)

No missing values found.
No infinite values found.
Duplicate rows found: 99955


# <font color = 'red'> Análisis

In [5]:
occupation_cols = [col for col in df.columns if col.startswith("Occupation_")]

results = counts_and_proportions_by_category(df, occupation_cols, "Credit_Mix")
counts_df = results["counts"]
proportions_df = results["proportions"]
summary_df = results["summary"]


counts_df.index = counts_df.index.str.replace('Occupation_', '')
proportions_df.index = proportions_df.index.str.replace('Occupation_', '')
summary_df.index = summary_df.index.str.replace('Occupation_', '')

In [10]:
custom_colors = {
    "prop_Bad": "orangered",
    "prop_Standard": "gold",
    "prop_Good": "skyblue"
}

fig = plot_categorical_proportions(
    df=proportions_df.reset_index(),  
    x_column="index",                 
    y_columns=["prop_Bad", "prop_Standard", "prop_Good"],  
    stacked=False,                      
    title="Proporción de Credit Score por Ocupación",
    x_title="",
    y_title="Proporción",
    legend_title="Credit Score",
    colors=custom_colors,              
    width=1000,
    height=600,
    sort_order='asc',          
    sort_by="prop_Bad"          
)

fig.show()


fig = plot_categorical_proportions(
    df=proportions_df.reset_index(),  
    x_column="index",                 
    y_columns=["prop_Bad"],  
    stacked=False,                      
    title="Proporción de Bad por Ocupación",
    x_title="",
    y_title="Proporción",
    legend_title="Credit Score",
    colors=custom_colors,              
    width=1000,
    height=400,
    sort_order='asc',          
    sort_by="prop_Bad"          
)

fig.show()

fig = plot_categorical_proportions(
    df=proportions_df.reset_index(),  
    x_column="index",                 
    y_columns=["prop_Good"],  
    stacked=False,                      
    title="Proporción Good por Ocupación",
    x_title="",
    y_title="Proporción",
    legend_title="Credit Score",
    colors=custom_colors,              
    width=1000,
    height=400,
    sort_order='asc',          
    sort_by="prop_Good"          
)

fig.show()

fig = plot_categorical_proportions(
    df=proportions_df.reset_index(),  
    x_column="index",                 
    y_columns=["prop_Standard"],  
    stacked=False,                      
    title="Proporción Standard por Ocupación",
    x_title="",
    y_title="Proporción",
    legend_title="Credit Score",
    colors=custom_colors,              
    width=1000,
    height=400,
    sort_order='asc',          
    sort_by="prop_Standard"          
)

fig.show()


<font color = 'skyblue'> Regresión

In [13]:
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel

# Definir la variable dependiente
y_column = "Credit_Score"

x_columns = [col for col in df.columns if "Occupation_" in col]

x_columns.remove("Occupation_Engineer")  

# Ajustar el modelo ordinal
model_occupation = OrderedModel(df[y_column], sm.add_constant(df[x_columns]), distr="logit")

# Ajustar el modelo
result_occupation = model_occupation.fit(method='bfgs')

# Mostrar los resultados
print(result_occupation.summary())


ValueError: There should not be a constant in the model

In [8]:
occupation_cols

['Occupation_Accountant',
 'Occupation_Architect',
 'Occupation_Developer',
 'Occupation_Doctor',
 'Occupation_Engineer',
 'Occupation_Entrepreneur',
 'Occupation_Journalist',
 'Occupation_Lawyer',
 'Occupation_Manager',
 'Occupation_Mechanic',
 'Occupation_Media_Manager',
 'Occupation_Musician',
 'Occupation_Scientist',
 'Occupation_Teacher',
 'Occupation_Writer']